In [1]:
import pandas as pd
from pathlib import Path
import h5py
import sys

reeds_path = '' # User should specify path to ReEDS repository here
sys.path.append(reeds_path)
import reeds

Especially on Linux, gdxpds should be imported before pandas to avoid a library conflict. Also make sure your GAMS directory is listed in LD_LIBRARY_PATH.


In [2]:
def get_temperatures(reeds_path, tz_in='UTC', tz_out='Etc/GMT+6'):
    h5path = Path(
        reeds_path, 'inputs', 'profiles_temperature', 'temperature_state.h5',
    )
    _temperatures = {}
    with h5py.File(h5path, 'r') as f:
        years = [int(i) for i in list(f) if i.isdigit()]
        for year in years:
            timeindex = pd.to_datetime(
                pd.Series(f[f"index_{year}"][:])
                .str.decode('utf-8')
            )
            _temperatures[year] = pd.DataFrame(
                index=timeindex,
                columns=pd.Series(f['columns']).map(lambda x: x.decode()),
                data=f[str(year)],
            )

    temperatures = (
        pd.concat(_temperatures, names=('year','timestamp')).rename_axis(columns='r')
        .reset_index('year', drop=True)
        .tz_localize(tz_in)
        .tz_convert(tz_out)
    )

    return temperatures

In [3]:
def calculate_daily_state_degree_days(temperatures):
    # Get baseline temperature for calculating degree days
    scalars = reeds.io.get_scalars()
    base_temp = scalars['degree_days_base_temperature']

    # Calculate degree-hours (hourly deviations from baseline)
    # and then take the daily averages of degree-hours to
    # get degree days. This is different from the traditional
    # approach for calculating degree days
    # (https://www.eia.gov/energyexplained/units-and-calculators/degree-days.php),
    # but we found that this approach generally gives better
    # regression model results (higher r-squared values and smaller errors). 
    hdd_hourly = (base_temp - temperatures).clip(lower=0)
    hdd_daily = hdd_hourly.resample('D').mean()

    cdd_hourly = (temperatures - base_temp).clip(lower=0)
    cdd_daily = cdd_hourly.resample('D').mean()
    

    return hdd_daily, cdd_daily

In [4]:
def aggregate_by_weighted_average(
    regional_data: pd.DataFrame,
    region_aggregion_weights: pd.Series,
    region2aggregion: dict[str, str]
) -> pd.DataFrame:
    """
    Aggregate region-level data to the aggregated region
    ("aggregion") level via weighted average.

    Args:
        regional_data: Region-level data.
        region_aggregion_weights: The "weight" of each region
            corresponding to its aggregion to use in weighted
            average calculation.
        region2aggregion: Mapping between regions and aggregions.

    Returns:
        pd.DataFrame
    """
    aggregional_data = (
        regional_data.mul(region_aggregion_weights)
        .transpose()
        .rename(region2aggregion)
        .groupby(level=0)
        .sum()
        .transpose()
    )
    return aggregional_data

In [5]:
def calculate_region_aggregion_population_weights(
    region_level: str,
    aggregion_level: str,
) -> pd.Series:
    """
    For a given region level and aggregated region (aggregion)
    level, calculate each region's share of its corresponding
    aggregion's total population.
    
    Args:
        inputs_case: Path to the inputs case directory.
        region_level: Region level (example: 'state')
        aggregion_level: Aggregated region level
            (example: 'cendiv')

    Returns:
        pd.Series
    """
    # Get county populations
    county_populations = reeds.inputs.get_county_populations()
    county_populations = county_populations.rename(
        columns={'value': 'population'}
    )

    # Get county-to-region mapping
    county2zone = reeds.io.get_county2zone(
        as_map=False
    )
    county2zone['FIPS'] = (
        'p' + county2zone['FIPS'].astype(str).str.zfill(5)
    )
    state_groups = reeds.inputs.get_state_groups()
    county2zone = county2zone.merge(
        state_groups,
        left_on='state',
        right_on='st'
    )
    county_region_map = county2zone.set_index('FIPS')[region_level]

    # Calculate regional populations
    county_populations[region_level] = (
        county_populations['FIPS'].map(county_region_map)
    )
    region_populations = (
        county_populations.groupby(region_level, as_index=False)
        ['population']
        .sum()
    )

    # Calculate each region's percentage of aggregion population
    region2aggregion = dict(zip(
        county2zone[region_level],
        county2zone[aggregion_level]
    ))
    region_populations[aggregion_level] = (
        region_populations[region_level].map(region2aggregion)
    )
    region_populations['weight'] = (
        region_populations['population']
        / (
            region_populations.groupby(aggregion_level)
            ['population']
            .transform('sum')
        )
    )
    region_aggregion_weights = (
        region_populations.set_index(region_level)['weight']
    )

    return region_aggregion_weights

In [6]:
# Get state-level HDD/CDDs
temperatures = get_temperatures(reeds_path)
hdd_daily_st, cdd_daily_st = calculate_daily_state_degree_days(temperatures)

# Get state-to-gasreg mapping
state_groups = reeds.inputs.get_state_groups()
st2gasreg = state_groups.set_index('st')['gasreg']

# Calculate population-based state-to-gasreg weights
state_gasreg_weights = calculate_region_aggregion_population_weights(
    region_level='state',
    aggregion_level='gasreg'
)

# Aggregate state-level HDD/CDDs via population-weighted
# average to get gasreg-level HDD/CDDs
hdd_daily_gasreg = aggregate_by_weighted_average(
    hdd_daily_st,
    state_gasreg_weights,
    st2gasreg
)
hdd_daily_gasreg = hdd_daily_gasreg.rename_axis(index='datetime')
cdd_daily_gasreg = aggregate_by_weighted_average(
    cdd_daily_st,
    state_gasreg_weights,
    st2gasreg
)
cdd_daily_gasreg = cdd_daily_gasreg.rename_axis(index='datetime')

In [7]:
# Get natural gas hub prices.
# Note this data is not accessible to people external to the lab.
hub_prices = pd.read_excel(
    '//nrelnas01/ReEDS/FY26_NatGas_KO/Natural Gas Daily Hub Prices - 07-17-2025 - Internal NREL only.xlsx'
)
hub_prices = hub_prices.loc[(
    hub_prices['Delivery Date'].dt.year.isin(range(2014, 2025))
)].copy()

In [8]:
# Assign hubs to gasregs. These are largely based on finding the geographic
# overlap between hub locations and gasregs in inspect_hub_locations.ipynb.
# In some cases, (e.g., East North Central), only a subset of the identified
# hub locations is chosen because it results in a better correlation (i.e.,
# higher HDD/CDD coefficients in the regression)
region_hub_map = {
    'California': ['California - North', 'California - South'],
    'East_North_Central': ['Chicago Metro'],
    'East_South_Central': ['Marcellus - Lower'],
    'Mid_Atlantic': ['Mid-Atlantic'],
    'Mountain': ['Green River'],
    'New_England': ['New England'],
    'Northwest': ['Northwest'],
    'South_Atlantic': ['South Atlantic', 'Mid-Atlantic'],
    'Southwest': ['Mojave'],
    'West_North_Central': ['Midcontinent - Upper', 'Midcontinent - Central'],
    'West_South_Central': ['Gulf Coast ELA', 'Gulf Coast ETX', 'TexOK']
}

In [9]:
# For each gasreg, calculate the volume-weighted average price of all hubs
def get_regional_volume_weighted_price(hub_prices, region, hub_names):
    hub_names = [f"Hitachi Energy {hub_name}" for hub_name in hub_names]
    select_hub_prices = (
        hub_prices
        .loc[hub_prices['Price Hub'].isin(hub_names)]
        .copy()
        .set_index('Delivery Date')
    )

    if len(hub_names) > 1:
        select_hub_prices['Total $'] = (
            select_hub_prices['Wtd Avg Index $'] * select_hub_prices['Daily Volume']
        )
        select_hub_prices = (
            select_hub_prices.groupby(level=0)
            .sum(numeric_only=True)
        )
        select_hub_prices['volume_weighted_price'] = (
            select_hub_prices['Total $'] / select_hub_prices['Daily Volume']
        )
        regional_volume_weighted_price = select_hub_prices['volume_weighted_price']
    else:
        regional_volume_weighted_price = select_hub_prices['Wtd Avg Index $']

    return regional_volume_weighted_price

regional_prices = {}
for region, hub_names in region_hub_map.items():
    regional_prices[region] = get_regional_volume_weighted_price(
        hub_prices,
        region,
        hub_names
    )

regional_prices = pd.concat(regional_prices, axis=1)
regional_prices = regional_prices.set_index([
    regional_prices.index.year,
    regional_prices.index.month,
    regional_prices.index.day
])
regional_prices = regional_prices.rename_axis(index=['year', 'month', 'day'])

In [10]:
# Combine daily HDD/CDD and gas price data
gasreg_cdd = cdd_daily_gasreg.set_index([
    cdd_daily_gasreg.index.year,
    cdd_daily_gasreg.index.month,
    cdd_daily_gasreg.index.day
])
gasreg_hdd = hdd_daily_gasreg.set_index([
    hdd_daily_gasreg.index.year,
    hdd_daily_gasreg.index.month,
    hdd_daily_gasreg.index.day
])

cdd_data = gasreg_cdd.loc[regional_prices.index].copy()
hdd_data = gasreg_hdd.loc[regional_prices.index].copy()

cdd_data.columns = [f"{col}_cdd" for col in cdd_data.columns]
hdd_data.columns = [f"{col}_hdd" for col in hdd_data.columns]

data = pd.concat([cdd_data, hdd_data], axis=1).fillna(0)
for region in region_hub_map.keys():
    data[f"{region}_price"] = regional_prices[region]

In [11]:
data = data.rename_axis(index=['year', 'month', 'day'])

In [12]:
# Export
outpath = Path('inputs', 'gasreg_regression_data.csv')
outpath.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(outpath)